# Segmentação de Imagens Acústicas de Poço sem *Ground Truth* (WAID)

**ELE2346 – Deep Learning** · Mestrado em Engenharia Elétrica (Sistemas Inteligentes) · PUC-Rio · 2026.2
Autor: Jorge Alberto Avelino de Souza

**Notebook 02 – Rótulos e *baselines* (Fase 2).** Dados: WAID (PETROBRAS, CC BY 4.0), 5 poços.

## Fase 2 – Rótulos e *baselines*
Saída: conjunto anotado por patch (40 patches), pseudo-rótulos de limiar, *baselines* e U-Net de referência.

### 2.1 Ambiente: bibliotecas e Google Drive
Importando as bibliotecas e montando o Drive.

In [1]:
# Importando as bibliotecas usadas no notebook
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive

In [2]:
# Montando o Drive para uso dos arquivos WAID
drive.mount("/content/drive")

Mounted at /content/drive


### 2.2 Configuração do projeto
Ajustando o caminho de busca e importando constantes e funções de `src`.

In [3]:
# Adicionando ao caminho de busca do Python a pasta que contém "src"
sys.path.insert(0, "/content/drive/MyDrive/waid")

# Importando as constantes (poços e pastas) e as funções do projeto
from src.config import POCOS, NPY_DIR, BSC_DIR, CARAC_DIR, ANOT_DIR, RES_DIR, ALTURA_PATCH
from src.labels.anotacao import (sortear_patches, sortear_segunda_rodada, abrir_interface,
                                 anotados, revisar_decisao, sortear_reanotacao, concordancia)
from src.labels.limiares import METODOS, limiar_otsu_poco, fracao_escura, calcular_limiares
from src.eval.patches import tabela_escores, avaliar

### 2.3 Conjunto anotado: sorteio dos patches
Sorteando 8 patches por poço (40 no total). O sorteio é salvo e não muda nas execuções seguintes.

In [4]:
# Definindo o número de patches por poço e a semente do sorteio
N_POR_POCO = 8
SEMENTE = 0

# Lendo o índice de patches válidos (decisão D4, sem as zonas da D1)
indice = pd.read_csv(f"{CARAC_DIR}/patches_indice.csv")

# Sorteando os patches uma só vez; nas execuções seguintes, relendo o sorteio salvo
os.makedirs(ANOT_DIR, exist_ok=True)
arq_sorteio = f"{ANOT_DIR}/sorteio.csv"
if os.path.exists(arq_sorteio):
    sorteio = pd.read_csv(arq_sorteio)
else:
    sorteio = sortear_patches(indice, N_POR_POCO, SEMENTE)
    sorteio.to_csv(arq_sorteio, index=False)
print(len(sorteio), "patches sorteados;", len(anotados(ANOT_DIR)), "já decididos")
print(sorteio.groupby("poco").size().to_string())

40 patches sorteados; 80 já decididos
poco
antilope25    8
antilope37    8
botorosa47    8
coala88       8
tatu22        8


### 2.4 Conjunto anotado: decisão por patch
Siga a ficha `docs/anotacao.md`. Em cada patch, responda uma pergunta só:

- **Feição escura clara:** há feição escura clara (*breakout*, fratura ou cavidade, pela ficha).
- **Sem feição:** não há zona escura clara.
- **Dúvida (patch sai):** qualquer incerteza. Faixas diagonais, linhas verticais finas e zonas escuras extensas também ficam como dúvida.

A curva CAL à direita é só evidência de apoio. Ela mostra se o poço está alargado. Decida uma vez. Não refaça um patch depois de ver o resultado do modelo. Se você parar, a lista continua de onde parou quando a célula for executada de novo.

In [5]:
# Abrindo a interface de anotação (mostra o primeiro patch ainda não decidido)
anotador = abrir_interface(sorteio, NPY_DIR, ANOT_DIR, BSC_DIR, ALTURA_PATCH)

### 2.5 Conjunto anotado: resumo
Contando as decisões por poço e comparando o CAL por decisão.

In [6]:
# Resumindo as decisões já salvas (linhas antigas, sem rodada, são da rodada 1)
tab = pd.read_csv(f"{ANOT_DIR}/anotacoes.csv")
if "rodada" not in tab.columns:
    tab.insert(0, "rodada", 1)
print(pd.crosstab(tab.poco, tab.decisao, margins=True))
print()

# Comparando o desvio robusto do CAL (cal_z) entre as decisões, nos patches com CAL
print(tab.groupby("decisao").cal_z.describe().round(2).to_string())

decisao     duvida  feicao_clara  sem_feicao  All
poco                                             
antilope25       1             9           6   16
antilope37       3             8           5   16
botorosa47       1            12           3   16
coala88          3            11           2   16
tatu22           1            15           0   16
All              9            55          16   80

              count  mean   std   min   25%   50%   75%    max
decisao                                                       
duvida          4.0  3.19  6.44 -0.38 -0.07  0.15  3.40  12.83
feicao_clara   32.0  0.61  1.07 -1.03 -0.09  0.33  1.08   3.23
sem_feicao     10.0  0.07  0.84 -0.83 -0.60 -0.27  0.82   1.54


### 2.6 Conjunto anotado: segunda rodada (sorteio dirigido)
A rodada 1 teve só 6 patches "sem feição". Na rodada 2, o coala88 e o tatu22 (sem negativos) recebem 8 patches do quartil mais baixo de área escura; os outros três poços recebem 4 desse quartil e 4 aleatórios. Isso aumenta a chance de "sem feição", mas introduz um **viés de seleção** que o artigo declara. Os patches novos não se sobrepõem aos da rodada 1.

In [7]:
# Calculando o limiar de Otsu de cada poço (1 a cada 10 linhas válidas)
limiares = {}
for p in POCOS:
    norm = np.load(f"{NPY_DIR}/{p}_norm.npy", mmap_mode="r")
    excl = np.load(f"{NPY_DIR}/{p}_excluir_rs.npy")
    limiares[p] = limiar_otsu_poco(norm, excl)
print(pd.Series(limiares).round(3).to_string())

# Calculando a fração de pixels escuros de cada patch válido e salvando a tabela
partes = []
for p, g in indice.groupby("poco"):
    g = g.copy()
    g["frac_escura"] = fracao_escura(NPY_DIR, p, g.linha_ini.values, limiares[p], ALTURA_PATCH)
    partes.append(g)
indice_fe = pd.concat(partes)
indice_fe.to_csv(f"{ANOT_DIR}/patches_fracao_escura.csv", index=False)

antilope25    0.541
antilope37    0.623
botorosa47    0.670
coala88       0.682
tatu22        0.760


In [8]:
# Sorteando a segunda rodada uma só vez; nas execuções seguintes, relendo o sorteio salvo
arq_sorteio2 = f"{ANOT_DIR}/sorteio_2.csv"
if os.path.exists(arq_sorteio2):
    sorteio2 = pd.read_csv(arq_sorteio2)
else:
    usados = pd.read_csv(f"{ANOT_DIR}/sorteio.csv")
    sorteio2 = sortear_segunda_rodada(indice_fe, usados, n_por_poco=8,
                                      n_baixa={"coala88": 8, "tatu22": 8}, n_baixa_padrao=4,
                                      semente=1, altura=ALTURA_PATCH)
    sorteio2.to_csv(arq_sorteio2, index=False)
print(len(sorteio2), "patches sorteados;", len(anotados(ANOT_DIR)), "decididos no total")
print(sorteio2.groupby(["poco", "origem"]).size().to_string())

40 patches sorteados; 80 decididos no total
poco        origem           
antilope25  aleatorio            4
            baixa_area_escura    4
antilope37  aleatorio            4
            baixa_area_escura    4
botorosa47  aleatorio            4
            baixa_area_escura    4
coala88     baixa_area_escura    8
tatu22      baixa_area_escura    8


### 2.7 Conjunto anotado: decisão por patch (rodada 2)
Siga a mesma ficha e as mesmas regras da seção 2.4. Decida uma vez.

In [9]:
# Abrindo a interface de anotação com o sorteio da rodada 2
anotador2 = abrir_interface(sorteio2, NPY_DIR, ANOT_DIR, BSC_DIR, ALTURA_PATCH)

### 2.8 Conjunto anotado: resumo final
Execute de novo a célula de resumo da seção 2.5 para ver as duas rodadas.

### 2.9 Conjunto anotado: revisão pela ficha
Dois patches de "feição clara" passam para "dúvida", pelos critérios da ficha. A decisão original e o motivo ficam registrados. A célula pode ser executada mais de uma vez.

In [10]:
# Passando para "dúvida" o patch dentro da zona de faixas diagonais do tatu22 (2382–2424 m)
revisar_decisao(ANOT_DIR, "tatu22", 41728, "duvida",
                "Revisão pela ficha: zona de faixas diagonais (2382–2424 m), causa não confirmada.")

# Passando para "dúvida" o patch com caliper muito alargado do antilope37 (cal_z = 12,8)
revisar_decisao(ANOT_DIR, "antilope37", 1792, "duvida",
                "Revisão pela ficha: caliper muito alargado (cal_z 12,8); possível washout.")

# Conferindo as decisões finais por poço
tab = pd.read_csv(f"{ANOT_DIR}/anotacoes.csv")
print(pd.crosstab(tab.poco, tab.decisao, margins=True))

decisao     duvida  feicao_clara  sem_feicao  All
poco                                             
antilope25       1             9           6   16
antilope37       3             8           5   16
botorosa47       1            12           3   16
coala88          3            11           2   16
tatu22           1            15           0   16
All              9            55          16   80


### 2.10 Conjunto anotado: reanotação às cegas
Sorteando 10 patches já decididos (4 do tatu22 e 6 de outros poços, metade "sem feição"). A interface **não mostra** a decisão anterior. Decida pela ficha, sem tentar lembrar a resposta. O resultado vai para `reanotacao.csv` e não altera `anotacoes.csv`. O intervalo de tempo entre as duas rodadas entra no artigo.

In [11]:
# Sorteando os 10 patches uma só vez; nas execuções seguintes, relendo o sorteio salvo
arq_rean = f"{ANOT_DIR}/sorteio_reanotacao.csv"
if os.path.exists(arq_rean):
    sorteio_rean = pd.read_csv(arq_rean)
else:
    tab = pd.read_csv(f"{ANOT_DIR}/anotacoes.csv")
    sorteio_rean = sortear_reanotacao(tab, {"tatu22": 4, "outros": 6}, semente=2)
    sorteio_rean.to_csv(arq_rean, index=False)
print(len(sorteio_rean), "patches para reanotar;", len(anotados(ANOT_DIR, "reanotacao.csv")), "já decididos")

10 patches para reanotar; 10 já decididos


In [12]:
# Abrindo a interface de reanotação (grava em reanotacao.csv)
anotador3 = abrir_interface(sorteio_rean, NPY_DIR, ANOT_DIR, BSC_DIR, ALTURA_PATCH, arquivo="reanotacao.csv")

### 2.11 Conjunto anotado: concordância entre as duas rodadas
Comparando a reanotação com a decisão original (só "feição clara" e "sem feição").

In [13]:
# Calculando a concordância simples e o kappa de Cohen entre as duas decisões
n, po, kappa, tabela = concordancia(ANOT_DIR)
print(f"patches comparados: {n} | concordância: {po:.2f} | kappa: {kappa:.2f}")
print(tabela)

patches comparados: 10 | concordância: 1.00 | kappa: 1.00
decisao       feicao_clara  sem_feicao
dec_a                                 
feicao_clara             7           0
sem_feicao               0           3


### 2.12 Baselines de limiar: avaliação por patch
Três limiares por poço, sobre a imagem normalizada: **Otsu** (2 classes), **Multi-Otsu** (3 classes; limiar da classe mais escura) e **K-means** (2 grupos; ponto médio entre os centros). O escore de cada patch é a fração de pixels abaixo do limiar. Contra as decisões "feição clara" e "sem feição" (71 patches), o notebook calcula:

- **AUC**, com intervalo de confiança de 95% por *bootstrap* (não depende de corte);
- **F1, precisão e revocação**, com o corte escolhido nos outros poços (*leave-one-well-out*);
- **AUC por poço**, só onde há as duas classes (o tatu22 não tem "sem feição").

In [14]:
# Calculando os limiares de cada poço e método e salvando a tabela
os.makedirs(RES_DIR, exist_ok=True)
lim = calcular_limiares(NPY_DIR, POCOS)
lim.to_csv(f"{RES_DIR}/limiares_baselines.csv")
print(lim.round(3).to_string())

             otsu  multiotsu3  kmeans2
poco                                  
antilope25  0.541       0.475    0.545
antilope37  0.623       0.455    0.622
botorosa47  0.670       0.490    0.666
coala88     0.682       0.447    0.682
tatu22      0.760       0.439    0.758


In [15]:
# Calculando a fração escura dos patches anotados (sem dúvida) e avaliando cada método
anot = pd.read_csv(f"{ANOT_DIR}/anotacoes.csv")
escores = tabela_escores(NPY_DIR, anot, lim, ALTURA_PATCH)
escores.to_csv(f"{RES_DIR}/baselines_limiar_escores.csv", index=False)
res = avaliar(escores, METODOS)
res.to_csv(f"{RES_DIR}/baselines_limiar_patches.csv", index=False)
print(res.T.to_string())

                    0           1        2
metodo           otsu  multiotsu3  kmeans2
n                  71          71       71
auc             0.932       0.911    0.932
ic95_baixo      0.856       0.818    0.857
ic95_alto       0.988       0.982    0.987
f1_lowo          0.92       0.939    0.919
precisao        0.897         0.9    0.911
revocacao       0.945       0.982    0.927
auc_antilope25  0.926       0.926    0.926
auc_antilope37  0.975       0.975    0.975
auc_botorosa47  0.972       0.972    0.972
auc_coala88       1.0         1.0      1.0
auc_tatu22        NaN         NaN      NaN
